# Phase 08 — Teaching Modes (Tutorial)

**Main question:** How do we shape what the VLM does with multimodal evidence?

In this tutorial we implement each teaching mode inline before switching to
the production `mrta.MultimodalRAG` implementation.

**Sections:**
1. Setup
2. Why teaching modes are not just prompt prefixes
3. Explain — inline implementation then production
4. Socratic, Quiz, Compare, Visual Evidence
5. Cross-validation

> **VLM requirement:** Sections 3-5 require Ollama with a vision model.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
from mrta import (
    Embedder, VectorStore, chunk_pdf, load_pdf,
    CLIPEmbedder, VisualVectorStore, extract_figures,
    MultimodalRetriever, VLMClient, MultimodalRAG,
)

SAMPLE_PDF = Path("../../tests/fixtures/sample.pdf")
assert SAMPLE_PDF.exists()

doc = load_pdf(SAMPLE_PDF)
chunks = chunk_pdf(doc)
embedder = Embedder()
text_store = VectorStore(embedder)
text_store.add(chunks)

clip = CLIPEmbedder()
figures = extract_figures(doc)
visual_records = [f.to_evidence_record() for f in figures]
visual_store = VisualVectorStore(clip)
if visual_records:
    visual_store.add(visual_records)

retriever = MultimodalRetriever(
    vector_store=text_store,
    visual_store=visual_store if visual_store.size > 0 else None,
    rrf_k=60,
)
vlm_available = VLMClient.is_available()
QUERY = "What is the role of the attention mechanism?"
print(f"Text: {text_store.size} | Figures: {len(visual_records)} | VLM: {vlm_available}")

---
## 08.2 Why Teaching Modes Are Not Prompt Prefixes

Text-only modes work by prepending an instruction:

```python
question = "Explain like I am new to this topic. " + question
```

This is weak: the LLM sees no labelled evidence, cannot reason about figures,
and the prefix instruction competes with the retrieved context.

Multimodal teaching modes work differently:

| Property | Prefix mode (text RAG) | Teaching mode (multimodal RAG) |
|---|---|---|
| Instruction location | prepended to question | in the prompt template |
| VLM sees images | no | yes |
| Evidence labelled | no | [T#] / [V#] |
| Retrieval changes | no | no (same pipeline) |

The retrieval is identical for all five modes.
Only the rendered Jinja2 template changes.

---
## 08.3 Explain Mode — Inline Then Production

We build the explain prompt manually to see every piece,
then cross-validate with `MultimodalRAG(teaching_mode='explain')`.

In [ ]:
def build_explain_prompt(
    question: str,
    text_evidence: list,
    visual_evidence: list,
) -> str:
    """Inline explain prompt — equivalent to teaching_explain.j2."""
    parts = [
        "You are a research and teaching assistant.",
        "Explain the topic using ONLY the evidence below at undergraduate level.",
        "Define terms when first used. Cite as [T1], [T2], [V1], [V2], etc.",
        "",
        "--- QUESTION ---",
        question,
        "",
    ]
    if text_evidence:
        parts.append("--- TEXT EVIDENCE ---")
        for i, ev in enumerate(text_evidence, 1):
            parts.append(f"[T{i}] {ev.source} | page {ev.page}")
            parts.append(ev.text or "")
            parts.append("")
    if visual_evidence:
        parts.append("--- VISUAL EVIDENCE ---")
        for i, ev in enumerate(visual_evidence, 1):
            fig_str = f" | Figure {ev.figure_index}" if ev.figure_index else ""
            parts.append(f"[V{i}] {ev.source} | page {ev.page}{fig_str}")
            parts.append("(image attached)" if ev.image_bytes else "(no image)")
            parts.append("")
    parts.append("--- EXPLANATION ---")
    return "\n".join(parts)


evidence = retriever.retrieve(QUERY, k_text=5, k_visual=5, k_final=8)
text_ev = [e for e in evidence if e.modality == "text"]
visual_ev = [e for e in evidence if e.modality in ("image", "page")]
images = [ev.to_pil() for ev in visual_ev if ev.image_bytes is not None]

inline_prompt = build_explain_prompt(QUERY, text_ev, visual_ev)
print(f"Prompt: {len(inline_prompt)} chars | Images: {len(images)}")

if vlm_available:
    vlm = VLMClient()
    answer_inline = vlm.generate(inline_prompt, images)
    print("\n--- EXPLAIN (inline) ---")
    print(answer_inline[:400])
else:
    print("VLM not available.")

In [ ]:
# Cross-validate with production MultimodalRAG
if vlm_available:
    mmrag = MultimodalRAG(
        retriever=retriever, vlm=vlm, teaching_mode="explain"
    )
    result = mmrag.ask(QUERY)
    print(f"mode={result.retrieval_mode} | {len(result.text_citations)}T + {len(result.visual_citations)}V")
    print("\n--- EXPLAIN (production) ---")
    print(result.answer[:400])
else:
    print("VLM not available.")

---
## 08.4 Socratic, Quiz, Compare, Visual Evidence

Each mode simply changes the `teaching_mode` argument.
The retrieval and image-passing logic is identical.

In [ ]:
if vlm_available:
    for mode in ["socratic", "quiz", "compare", "visual_evidence"]:
        mmrag_mode = MultimodalRAG(
            retriever=retriever, vlm=vlm, teaching_mode=mode
        )
        r = mmrag_mode.ask(QUERY)
        print(f"=== {mode.upper()} (mode={r.retrieval_mode}) ===")
        print(f"Citations: {[c.label for c in r.text_citations]} text, "
              f"{[c.label for c in r.visual_citations]} visual")
        print(r.answer[:300])
        print()
else:
    print("VLM not available.")

---
## Summary

| Tutorial concept | Production equivalent |
|---|---|
| `build_explain_prompt()` | `teaching_explain.j2` |
| `vlm.generate(prompt, images)` | `MultimodalRAG.ask()` |
| `teaching_mode=` parameter | selects Jinja2 template |

**Key takeaway:**
All five teaching modes share one retrieval pipeline.
The teaching mode controls only which Jinja2 template the VLM receives.
Switching modes costs nothing computationally.

> Phase 09 evaluates these modes quantitatively.